# Connect to Genie API

| Step | What you'll do |
|------|---------------|
| **Step 1** | Connect to the Genie API and ask a question |
| **Step 2** | Wrap it in a simple orchestrator (1 Genie) |
| **Step 3** | Scale to multiple Genie Spaces |

In [0]:
%pip install -qU databricks-langchain

dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## Imports

In [0]:
import json
import time
from datetime import timedelta
from typing import Any  # for type hints

import mlflow
import pandas as pd
from databricks.sdk import WorkspaceClient  # Connection to Databricks SDK + authentication
from databricks.sdk.service.dashboards import GenieMessage  # Response type from Genie
from databricks_langchain import ChatDatabricks
from IPython.display import Markdown, display

## Configure Variables

* genie_space_id: str | Genie Space -> Configure -> Settings -> Space ID

In [0]:
dbutils.widgets.text("genie_space_id", "01f0fe3570de1a69a78b1b9b71d392ff", "1. Genie Space ID")

### Save to global variables

In [0]:
GENIE_SPACE_ID = dbutils.widgets.get("genie_space_id")

print(GENIE_SPACE_ID)

01f0fe3570de1a69a78b1b9b71d392ff


## Step 1: Connect to Genie through Workspace SDK

* Use the [WorkspaceClient library](https://databricks-sdk-py.readthedocs.io/en/latest/workspace/dashboards/genie.html$0) with for U2M authentication
* Create a conversation and parse the response with `print_output(message)`

In [0]:
# Fill in w
w = WorkspaceClient()

In [0]:
# Fill in msg
msg = w.genie.start_conversation_and_wait(
    space_id=GENIE_SPACE_ID, content="What are the top 10 vehicles by total order value?"
)
print(msg)

GenieMessage(id='01f0fe483df115ae95283c5e8200c1b8', space_id='01f0fe3570de1a69a78b1b9b71d392ff', conversation_id='01f0fe483dea1298bbdecdbf492daaa0', content='What are the top 10 vehicles by total order value?', message_id='01f0fe483df115ae95283c5e8200c1b8', attachments=[GenieAttachment(attachment_id='01f0fe4840991dd5b1d039fc966229f9', query=GenieQueryAttachment(description='You want to see the top 10 vehicles ranked by the total value of their completed orders.', id=None, last_updated_timestamp=None, parameters=[], query="SELECT\n  o.vehicle_id,\n  v.make,\n  v.model,\n  v.year,\n  SUM(oi.total_price) AS total_order_value\nFROM sandbox.sales.orders o\nJOIN sandbox.sales.order_items oi ON o.order_id = oi.order_id\nJOIN sandbox.sales.vehicles v ON o.vehicle_id = v.vehicle_id\nWHERE oi.item_type = 'Vehicle'\n  AND o.vehicle_id IS NOT NULL\n  AND oi.total_price IS NOT NULL\nGROUP BY o.vehicle_id, v.make, v.model, v.year\nORDER BY total_order_value DESC\nLIMIT 10", query_result_metadata=Gen

### GenieMessage Structure

`start_conversation_and_wait` returns a `GenieMessage` with these key fields:

| Field | Type | Description |
|-------|------|-------------|
| `content` | `str` | The original question you sent |
| `conversation_id` | `str` | Reuse this for follow-up questions |
| `message_id` | `str` | Unique ID for this message |
| `attachments` | `List[GenieAttachment]` | AI-generated response — SQL, text answer, and/or suggested follow-ups |
| `status` | `MessageStatus` | `COMPLETED`, `FAILED`, etc. |
| `error` | `MessageError \| None` | Error details if status is `FAILED` |

**Attachments** are the interesting part — each one can contain:
- `.query` — the generated SQL and its description
- `.text` — the natural-language answer
- `.suggested_questions` — follow-up question suggestions

Not every attachment has all three; check for `None` before accessing.

In [0]:
# Safely extract the text answer from attachments
answer = next((att.text.content for att in msg.attachments if att.text and att.text.content), None)
print(answer)

The top 10 vehicles by total order value are all **Mercedes-Benz** models, with total order values ranging from **$4,825,872** to **$4,315,052**. Notable vehicles include:
- **2026 GLC**: $4,825,872
- **2026 C-Class**: $4,754,347
- **2024 GLE**: $4,753,224
- **2024 C-Class**: $4,677,652
- **2025 C-Class**: $4,614,112

All top vehicles are recent models (2022-2026), and the GLC and C-Class appear most frequently in the top rankings.


### Parse Message

## Helper Functions

| Method | Purpose |
|--------|---------|
| `print_output(message)` | Format a `GenieMessage` into a readable dict (question, SQL, answer) |
| `ask_genie(query, space_id)` | Send a question to a Genie Space and return a parsed result dict |
| `get_space_metadata(space_id)` | Fetch table schemas, sample questions, and descriptions for a space |
| `to_dataframe(result)` | Convert a Genie result into a `pandas.DataFrame` via `get_message_attachment_query_result` |
| `ask_genie_verbose(query, space_id)` | Like `ask_genie` but prints each status transition during polling |

In [0]:
class GenieUtils:
    @staticmethod
    def print_output(message: GenieMessage) -> None:
        """Pretty-print a GenieMessage: question, SQL, and answer."""
        sql = None
        answer = None
        for att in message.attachments or []:
            if att.query and att.query.query and sql is None:
                sql = att.query.query
            if att.text and att.text.content and answer is None:
                answer = att.text.content

        print(f"Question:\n  {message.content}\n")
        if sql:
            print(f"SQL:\n  {sql}\n")
        if answer:
            print(f"Answer:\n  {answer}")

    @staticmethod
    def ask_genie(query: str, space_id: str, client: WorkspaceClient = None) -> dict:
        if client is None:
            raise ValueError("client (WorkspaceClient) is required")

        msg = client.genie.start_conversation_and_wait(
            space_id=space_id,
            content=query,
            timeout=timedelta(seconds=120),
        )

        msg_dict = msg.as_dict() if hasattr(msg, "as_dict") else vars(msg)

        sql, description, text = None, None, None
        text_contents = []

        for att in msg_dict.get("attachments") or []:
            if att.get("query"):
                sql = sql or att["query"].get("query")
                description = description or att["query"].get("description")
            if att.get("text"):
                content = att["text"].get("content")
                if content:
                    text_contents.append(content)

        text = max(text_contents, key=len) if text_contents else None

        return {"message": msg, "sql": sql, "description": description, "data": None, "text": text}

    @staticmethod
    def to_dataframe(result: dict, client: WorkspaceClient = None) -> "pd.DataFrame":
        """Convert a Genie result dict into a pandas DataFrame.

        Uses get_message_attachment_query_result to fetch the tabular data
        from the first query attachment. Falls back to a single-column
        DataFrame containing the text answer if no query attachment exists.
        """
        if client is None:
            raise ValueError("client (WorkspaceClient) is required")

        msg = result["message"]
        msg_dict = msg.as_dict() if hasattr(msg, "as_dict") else vars(msg)

        # Find the first query attachment with a statement_id
        for att in msg_dict.get("attachments") or []:
            query_att = att.get("query")
            if query_att and query_att.get("statement_id"):
                qr = client.genie.get_message_attachment_query_result(
                    space_id=msg.space_id,
                    conversation_id=msg.conversation_id,
                    message_id=msg.id,
                    attachment_id=att["attachment_id"],
                )
                # qr is a GenieGetMessageQueryResultResponse
                # Access: qr.statement_response.manifest.schema.columns / .result.data_array
                stmt = qr.statement_response
                columns = [c.name for c in (stmt.manifest.schema.columns or [])]
                rows = stmt.result.data_array or []
                return pd.DataFrame(rows, columns=columns)

        # Fallback: return the text answer as a single-row DataFrame
        return pd.DataFrame([{"answer": result.get("text", "")}])

    @staticmethod
    def ask_genie_verbose(query: str, space_id: str, client: WorkspaceClient = None) -> dict:
        """Query Genie with verbose status output.

        Uses the lower-level start_conversation + polling loop so you can
        see each status transition (ASKING_AI -> EXECUTING_QUERY -> COMPLETED).
        Returns the same dict format as ask_genie.
        """
        if client is None:
            raise ValueError("client (WorkspaceClient) is required")

        # Start conversation (non-blocking)
        conv = client.genie.start_conversation(
            space_id=space_id,
            content=query,
        )
        conversation_id = conv.conversation_id
        message_id = conv.message_id
        print(f"[verbose] Conversation started: {conversation_id}")
        print(f"[verbose] Message ID: {message_id}")

        last_status = None
        while True:
            msg = client.genie.get_message(
                space_id=space_id,
                conversation_id=conversation_id,
                message_id=message_id,
            )
            current_status = msg.status.value if msg.status else "UNKNOWN"
            if current_status != last_status:
                print(f"[verbose] Status: {current_status}")
                last_status = current_status

            if current_status in ("COMPLETED", "FAILED"):
                break
            time.sleep(2)

        if current_status == "FAILED":
            error = msg.error if hasattr(msg, "error") else "Unknown error"
            print(f"[verbose] Genie returned an error: {error}")
            return {"message": msg, "sql": None, "description": None, "data": None, "text": None}

        # Parse the completed message (same logic as ask_genie)
        msg_dict = msg.as_dict() if hasattr(msg, "as_dict") else vars(msg)
        sql, description, text = None, None, None
        text_contents = []

        for att in msg_dict.get("attachments") or []:
            if att.get("query"):
                sql = sql or att["query"].get("query")
                description = description or att["query"].get("description")
            if att.get("text"):
                content = att["text"].get("content")
                if content:
                    text_contents.append(content)

        text = max(text_contents, key=len) if text_contents else None
        print(f"[verbose] Done — SQL: {'yes' if sql else 'no'}, Text: {'yes' if text else 'no'}")

        return {"message": msg, "sql": sql, "description": description, "data": None, "text": text}

    @staticmethod
    def get_space_metadata(space_id: str, client: WorkspaceClient = None, enrich_columns: bool = False) -> dict:
        """Fetch metadata for a Genie Space.

        Args:
            enrich_columns: When True, fetches full column schemas from
                Unity Catalog for every table in the space. When False,
                only returns manually-annotated column_configs from the
                Genie Space configuration.
        """
        if client is None:
            raise ValueError("client (WorkspaceClient) is required")

        space = client.genie.get_space(
            space_id=space_id,
            include_serialized_space=True,
        )

        serialized = space.serialized_space
        if not serialized:
            raise RuntimeError("serialized_space missing in response (check permissions: need CAN EDIT on the space)")

        cfg = json.loads(serialized)
        config = cfg.get("config", {})
        data_sources = cfg.get("data_sources", {})

        sample_questions = [" ".join(q.get("question", [])) for q in config.get("sample_questions", [])]

        tables = []
        for t in data_sources.get("tables", []):
            identifier = t.get("identifier")
            # Genie Space column annotations (manually configured)
            genie_columns = [
                {
                    "column_name": col_meta.get("column_name"),
                    "description": col_meta.get("description"),
                }
                for col_meta in t.get("column_configs", [])
            ]

            # Optionally enrich with Unity Catalog schema
            uc_columns = []
            if enrich_columns and identifier:
                try:
                    uc_table = client.tables.get(full_name=identifier)
                    uc_columns = [
                        {
                            "column_name": col.name,
                            "type": str(col.type_name.value) if col.type_name else None,
                            "comment": col.comment,
                        }
                        for col in (uc_table.columns or [])
                    ]
                except Exception as e:
                    uc_columns = [{"error": str(e)}]

            table_info = {
                "identifier": identifier,
                "description": t.get("description"),
                "columns": uc_columns if enrich_columns else genie_columns,
            }
            tables.append(table_info)

        return {
            "space_id": space.space_id,
            "title": space.title,
            "description": space.description,
            "sample_questions": sample_questions,
            "tables": tables,
        }

In [0]:
GenieUtils.print_output(msg)

Question:
  What are the top 10 vehicles by total order value?

SQL:
  SELECT
  o.vehicle_id,
  v.make,
  v.model,
  v.year,
  SUM(oi.total_price) AS total_order_value
FROM sandbox.sales.orders o
JOIN sandbox.sales.order_items oi ON o.order_id = oi.order_id
JOIN sandbox.sales.vehicles v ON o.vehicle_id = v.vehicle_id
WHERE oi.item_type = 'Vehicle'
  AND o.vehicle_id IS NOT NULL
  AND oi.total_price IS NOT NULL
GROUP BY o.vehicle_id, v.make, v.model, v.year
ORDER BY total_order_value DESC
LIMIT 10

Answer:
  The top 10 vehicles by total order value are all **Mercedes-Benz** models, with total order values ranging from **$4,825,872** to **$4,315,052**. Notable vehicles include:
- **2026 GLC**: $4,825,872
- **2026 C-Class**: $4,754,347
- **2024 GLE**: $4,753,224
- **2024 C-Class**: $4,677,652
- **2025 C-Class**: $4,614,112

All top vehicles are recent models (2022-2026), and the GLC and C-Class appear most frequently in the top rankings.


### Multi-Turn Conversations

Every Genie response includes a `conversation_id`. Passing it back with `create_message_and_wait` lets you ask follow-up questions in the same conversation — Genie remembers the SQL context, so "break that down by region" works without restating the original question.

In [0]:
# Get the conversation ID from the first message
conversation_id = msg.conversation_id
print(f"Conversation ID: {conversation_id}")

Conversation ID: 01f0fe483dea1298bbdecdbf492daaa0


In [0]:
# Follow up in the same conversation — Genie remembers context

followup = w.genie.create_message_and_wait(
    space_id=GENIE_SPACE_ID,
    conversation_id=conversation_id,
    content="Break that down by customer segment",
    timeout=timedelta(seconds=120),
)
GenieUtils.print_output(followup)

Question:
  Break that down by customer segment

SQL:
  SELECT
  o.vehicle_id,
  v.make,
  v.model,
  v.year,
  cs.segment_name,
  SUM(oi.total_price) AS total_order_value
FROM sandbox.sales.orders o
JOIN sandbox.sales.order_items oi ON o.order_id = oi.order_id
JOIN sandbox.sales.vehicles v ON o.vehicle_id = v.vehicle_id
JOIN sandbox.crm.customers c ON o.customer_id = c.customer_id
JOIN sandbox.crm.customer_segments cs ON c.segment_id = cs.segment_id
WHERE oi.item_type = 'Vehicle'
  AND o.vehicle_id IS NOT NULL
  AND oi.total_price IS NOT NULL
  AND cs.segment_name IS NOT NULL
GROUP BY o.vehicle_id, v.make, v.model, v.year, cs.segment_name
ORDER BY total_order_value DESC
LIMIT 10

Answer:
  All of the top 10 vehicles by total order value were purchased by customers in the **Individual** segment. Notable vehicles and their total order values include:
- **2026 Mercedes-Benz C-Class**: $3,770,689
- **2025 Mercedes-Benz E-Class**: $3,658,452
- **2024 Mercedes-Benz C-Class**: $3,621,408
- *

### Status Transitions

When Genie processes a question it moves through a lifecycle:

| Status | Meaning |
|--------|---------|
| `SUBMITTED` | Request received |
| `ASKING_AI` | LLM is generating SQL |
| `EXECUTING_QUERY` | SQL is running on the warehouse |
| `COMPLETED` | Results are ready |

`ask_genie_verbose` uses the low-level `start_conversation` + `get_message` polling loop so you can watch each transition in real time.

In [0]:
# Verbose mode: see each status transition as Genie processes your question
GenieUtils.ask_genie_verbose("Which salesperson has the highest total sales this year?", GENIE_SPACE_ID, client=w)

[verbose] Conversation started: 01f0fe484d0b174b99eeb9fed5886675
[verbose] Message ID: 01f0fe484d111457b9f3e6e0e6966750
[verbose] Status: SUBMITTED
[verbose] Status: ASKING_AI
[verbose] Status: PENDING_WAREHOUSE
[verbose] Status: ASKING_AI
[verbose] Status: COMPLETED
[verbose] Done — SQL: yes, Text: yes


{'message': GenieMessage(id='01f0fe484d111457b9f3e6e0e6966750', space_id='01f0fe3570de1a69a78b1b9b71d392ff', conversation_id='01f0fe484d0b174b99eeb9fed5886675', content='Which salesperson has the highest total sales this year?', message_id='01f0fe484d111457b9f3e6e0e6966750', attachments=[GenieAttachment(attachment_id='01f0fe484f501677a272568eb73f6da2', query=GenieQueryAttachment(description='You want to see the salesperson with the highest total sales for the current year.', id=None, last_updated_timestamp=None, parameters=[], query="SELECT s.name AS salesperson, SUM(o.order_total) AS total_sales\nFROM `sandbox`.`sales`.`orders` o\nJOIN `sandbox`.`sales`.`salespersons` s ON o.salesperson_id = s.salesperson_id\nWHERE o.status = 'Completed' AND YEAR(o.order_date) = YEAR(CURRENT_DATE)\nGROUP BY s.salesperson_id, s.name\nORDER BY total_sales DESC\nLIMIT 1", query_result_metadata=GenieResultMetadata(is_truncated=None, row_count=1), statement_id='01f0fe48-4f59-14b7-8f1e-40d69f866e84', title=

### Pandas Display

`GenieUtils.to_dataframe(result)` calls `get_message_attachment_query_result` under the hood to fetch the raw rows from the query attachment, then wraps them in a `pandas.DataFrame` for easy display and downstream analysis.

In [0]:
# Convert Genie result to a pandas DataFrame
result = GenieUtils.ask_genie("What are the total orders by region?", GENIE_SPACE_ID, client=w)
df = GenieUtils.to_dataframe(result, client=w)
df

,region_name,total_orders
0,Mountain,10311
1,Southwest,10306
2,Pacific,10261
3,Mid-Atlantic,10191
4,Midwest,10170
5,Southeast,9980
6,Northeast,8407
7,Great Plains,8382
8,Gulf Coast,6772


## Connect to LLM
* Use ChatDatabricks to connect to an endpoint - for a list view AI/ML -> Serving in the tab on the left hand side
* Bonus - Set up MLFlow autologging

In [0]:
mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")
mlflow.langchain.autolog()

In [0]:
llm = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", temperature=1)

In [0]:
llm.invoke("How is the weather today?")

AIMessage(content='I don’t have real-time weather data. If you tell me your location (city or ZIP), I can give you a general forecast approach or look up the current conditions if you’d like. Alternatively, you can check a local weather app or website (e.g., Weather.com, AccuWeather, or your device’s weather app) for up-to-the-minute conditions. Would you like me to guide you on finding this quickly?', additional_kwargs={}, response_metadata={'usage': {'prompt_tokens': 12, 'completion_tokens': 96, 'total_tokens': 108}, 'prompt_tokens': 12, 'completion_tokens': 96, 'total_tokens': 108, 'model': 'gpt-5-nano-2025-08-07', 'model_name': 'gpt-5-nano-2025-08-07', 'finish_reason': 'stop'}, id='lc_run--019c11c5-fa4d-7ee0-9ea7-0ae4222313cb-0', tool_calls=[], invalid_tool_calls=[])

Trace(trace_id=tr-04fee2a5920ee35c52b18cfb632d834a)

## Step 2: Orchestrator (Single Genie)

Now that we can query Genie and parse its response, let's wrap everything into an **orchestrator** — a thin layer that:
1. Reads space metadata to understand available tables
2. Uses an LLM to rewrite and route the user's question
3. Sends the improved query to the right Genie Space
4. Formats the result for the user

In [0]:
GenieUtils.ask_genie("How many orders were placed last month by payment method?", GENIE_SPACE_ID, client=w)

{'message': GenieMessage(id='01f0fe485a8f13d0ba5ffec9c5f3ea8d', space_id='01f0fe3570de1a69a78b1b9b71d392ff', conversation_id='01f0fe485a891c75a4bc3ab3fc9153f6', content='How many orders were placed last month by payment method?', message_id='01f0fe485a8f13d0ba5ffec9c5f3ea8d', attachments=[GenieAttachment(attachment_id='01f0fe485e601068859895990a678798', query=GenieQueryAttachment(description='You want to see the number of orders placed last month, broken down by payment method.', id=None, last_updated_timestamp=None, parameters=[], query="SELECT\n  o.payment_method,\n  COUNT(*) AS order_count\nFROM sandbox.sales.orders o\nWHERE o.order_date >= date_trunc('MONTH', dateadd(MONTH, -1, CURRENT_DATE))\n  AND o.order_date < date_trunc('MONTH', CURRENT_DATE)\n  AND o.payment_method IS NOT NULL\nGROUP BY o.payment_method\nORDER BY order_count DESC", query_result_metadata=GenieResultMetadata(is_truncated=None, row_count=4), statement_id='01f0fe48-5e67-1516-b0c0-856a6dea958e', title=None), sugge

### Space Metadata for Routing

`get_space_metadata` returns the table schemas, column descriptions, and sample questions configured in a Genie Space. The router LLM needs this context to decide **which space** can answer a question and to **rewrite** the query so Genie produces the best SQL.

In [0]:
# enrich_columns=True fetches full schemas from Unity Catalog
GenieUtils.get_space_metadata(GENIE_SPACE_ID, client=w, enrich_columns=True)

{'space_id': '01f0fe3570de1a69a78b1b9b71d392ff',
 'title': 'Velocity Motors - Unified Analytics',
 'description': 'Cross-domain AI analytics for sales, customer, and operations data across the entire dealership network',
 'sample_questions': ['Which customer segments generate the most service revenue?',
  'Average service revenue per vehicle by make and model?',
  'Which salespeople have the highest customer retention based on service orders?',
  'Which vehicles have the highest service costs relative to sale price?',
  'Show price history for vehicles that had more than 2 price changes',
  'Top 10 customers by combined vehicle purchases and service spending?',
  'Customer satisfaction ratings by segment',
  'Which vehicles have both sunroof and navigation features?',
  'Show total sales by territory and region with division rollup',
  'What is the lead conversion rate by region?',
  'How many direct reports does each sales manager have?',
  'What are the most popular vehicle features 

### LLM Router

`ask_router` takes the user's question plus a list of Genie Space IDs. For each space it fetches metadata (tables, columns, sample questions), then asks the LLM to:
1. **Rewrite** the question to match the available schema
2. **Pick** the best space to answer it

The output is a dict with `{"query": "...", "space_id": "..."}`.

In [0]:
from pydantic import BaseModel, Field

class RouterResponse(BaseModel):
    """Structured response from the routing LLM."""
    query: str = Field(description="The improved version of the question, rewritten to be maximally useful to Genie")
    space_id: str = Field(description="The genie space ID to which to route the question")

def ask_router(
    llm: ChatDatabricks,
    question: str,
    genie_spaces: list[str],
    client: WorkspaceClient = None,
) -> dict[str, Any]:
    prompt = f"""
You are an intelligent router and orchestrator for Databricks Genie AI, an agent proficient in natural language -> SQL generation and retrieval. You are given a question from a user, which may be incomplete or inaccurate relative to the data available, and your task is to improve and augment the question to be maximally useful to Genie.

Use best practices and your knowledge of the available tables to improve the question to elicit the most effective response from Genie possible.

Question: {question}
"""
    # Append space metadata so the LLM knows what data is available
    prompt += "\n\nGenie Spaces Available and their metadata\n"
    for space_id in genie_spaces:
        space_metadata = GenieUtils.get_space_metadata(space_id, client=client, enrich_columns=True)
        prompt += f"Space ID: {space_id}\nMetadata: {space_metadata}\n"

    # Use structured output to guarantee valid JSON
    structured_llm = llm.with_structured_output(RouterResponse)
    result = structured_llm.invoke(prompt)

    return result.model_dump()


In [0]:
router_response = ask_router(llm, "Which vehicle models have the most service orders?", [GENIE_SPACE_ID], client=w)

print(router_response)

{'query': "From the vehicles table, identify model-level vehicle models with the highest number of service orders. Return model name, make, count of associated service orders, and average service order total cost. Consider service orders joined on vehicle_id, filter for completed orders, and group by make, model. Sort by count descending and show top 10 models. Optional: include the total revenue from service orders for those models and the median service duration. Include only vehicles currently in inventory (status = 'Available' or 'In Transit').", 'space_id': '01f0fe3570de1a69a78b1b9b71d392ff'}


Trace(trace_id=tr-c1d83fb5323376ff5a930cfef121fcef)

### GenieOrchestrator Class

`GenieOrchestrator` wires the three pieces together:
- **Router** — rewrites the question and picks the right Genie Space
- **Genie** — executes the query via `ask_genie`
- **LLM formatter** — produces a human-friendly summary of the result

Call `orchestrator.execute(question)` to run the full pipeline.

In [0]:
class GenieOrchestrator:
    def __init__(
        self,
        llm,
        space_ids: list[str] = None,
        client: WorkspaceClient = None,
    ):
        self.llm = llm
        self.space_ids = list(space_ids) if space_ids else [GENIE_SPACE_ID]
        self.client = client

        # Cache space titles at init time (1 lightweight API call per space)
        self.space_titles = {}
        for sid in self.space_ids:
            self._cache_title(sid)

    def _cache_title(self, space_id: str):
        """Fetch and cache the title for a single space."""
        try:
            space = self.client.genie.get_space(space_id=space_id)
            self.space_titles[space_id] = space.title
        except Exception:
            self.space_titles[space_id] = space_id

    def execute(self, user_question: str, output_format: str = "full"):
        """Run the full orchestration pipeline.

        Args:
            output_format: How to display results.
                "full" — print question, SQL, and answer (default, no LLM cost)
                "text" — print only the text answer
                "llm"  — use the LLM to produce a polished summary
                "raw"  — no printing, just return the dict
        """
        # determine genie space id and enhanced query
        router_response = ask_router(
            llm=self.llm,
            question=user_question,
            genie_spaces=self.space_ids,
            client=self.client,
        )

        routed_id = router_response["space_id"]
        routed_title = self.space_titles.get(routed_id, routed_id)
        print(f"Routed to space: {routed_title} ({routed_id})")
        print(f"Rewritten query: {router_response['query']}\n")

        genie_response = GenieUtils.ask_genie(
            query=router_response["query"],
            space_id=routed_id,
            client=self.client,
        )

        if output_format == "full":
            print(f"Question:\n  {user_question}\n")
            if genie_response.get("sql"):
                print(f"SQL:\n  {genie_response['sql']}\n")
            if genie_response.get("text"):
                print(f"Answer:\n  {genie_response['text']}")
        elif output_format == "text":
            print(genie_response.get("text", "No text answer returned."))
        elif output_format == "llm":
            summary = self.llm.invoke(
                f"Format the following Genie response into a clean, readable summary for the user:\n{genie_response}"
            )
            display(Markdown(summary.content))
        # "raw" — no printing

        return genie_response

    def add_space(self, space_id: str):
        self.space_ids.append(space_id)
        self._cache_title(space_id)

In [0]:
# Single-space orchestrator — defaults to GENIE_SPACE_ID when no list is passed
orch = GenieOrchestrator(
    llm=llm,
    client=w,
)

In [0]:
orch.execute("What are the top 5 selling vehicle models by total revenue?", "llm")

Routed to space: Velocity Motors - Unified Analytics (01f0fe3570de1a69a78b1b9b71d392ff)
Rewritten query: Identify the top 5 vehicle models by total revenue from vehicle sales. Use the sales data to compute total revenue per vehicle model by summing order totals for completed orders that include a vehicle, grouped by vehicle make and model, and sorted by revenue descending. Include model name, make, total revenue, number of units sold, and the time period if available. Exclude vehicles without corresponding orders. If possible, filter to the most recent full year and show currency in USD.



Here’s a clean summary of the request and results:

Summary of request:
- Objective: Identify the top 5 vehicle models by total revenue from completed vehicle sales.
- Method: Sum order totals for completed orders that include a vehicle, grouped by make and model, excluding vehicles without orders.
- Period: Prefer the most recent full year, with the period displayed.
- Output details: Include model name, make, total revenue (USD), units sold, and the period covered.
- Currency: USD.

Current results:
- Time period: January 1, 2026 to December 31, 2026 (most recent full year available in the data).
- Top 5 vehicle models by total revenue (USD) and units sold:
  1) Mercedes-Benz C-Class
     - Make: Mercedes-Benz
     - Model: C-Class
     - Total revenue: $12,880,216
     - Units sold: 117
     - Period: 2026-01-01 to 2026-12-31
  2) Mercedes-Benz E-Class
     - Make: Mercedes-Benz
     - Model: E-Class
     - Total revenue: $8,865,183
     - Units sold: 84
     - Period: 2026-01-01 to 2026-12-31
  3) Mercedes-Benz GLC
     - Make: Mercedes-Benz
     - Model: GLC
     - Total revenue: $7,721,719
     - Units sold: 88
     - Period: 2026-01-01 to 2026-12-31
  4) BMW 3 Series
     - Make: BMW
     - Model: 3 Series
     - Total revenue: $6,590,554
     - Units sold: 95
     - Period: 2026-01-01 to 2026-12-31
  5) Mercedes-Benz S-Class
     - Make: Mercedes-Benz
     - Model: S-Class
     - Total revenue: $6,315,755
     - Units sold: 105
     - Period: 2026-01-01 to 2026-12-31

Notes:
- All figures are for completed orders within the specified period.
- Mercedes-Benz models dominate the top revenue spots in this dataset for 2026.
- If you’d prefer a rolling 12-month view instead of the calendar year, I can adjust the query accordingly.

{'message': GenieMessage(id='01f0fe48664214ff805d46cf198a4f59', space_id='01f0fe3570de1a69a78b1b9b71d392ff', conversation_id='01f0fe48663d1533a13580cf57888586', content='Identify the top 5 vehicle models by total revenue from vehicle sales. Use the sales data to compute total revenue per vehicle model by summing order totals for completed orders that include a vehicle, grouped by vehicle make and model, and sorted by revenue descending. Include model name, make, total revenue, number of units sold, and the time period if available. Exclude vehicles without corresponding orders. If possible, filter to the most recent full year and show currency in USD.', message_id='01f0fe48664214ff805d46cf198a4f59', attachments=[GenieAttachment(attachment_id='01f0fe486b8a1c789d25373e0367431f', query=GenieQueryAttachment(description='You want to see the top 5 vehicle models ranked by total revenue from vehicle sales for the most recent full year, including the make, model, total revenue in USD, number o

[Trace(trace_id=tr-0cab9213c83f0f177b6257ae8d5a1584), Trace(trace_id=tr-5f39b8a6f638c739ae97dbdcfacb1be0)]

## Multi Genie Orchestration

Real organisations split data across multiple Genie Spaces (Sales, Customers, Inventory, etc.).

An orchestrator queries **all spaces** and combines the results.

```
Question
   |
   +---> Sales Genie      ---> revenue data
   +---> Customers Genie   ---> segment data
   +---> Inventory Genie   ---> stock data
   |
   v
Combined Results
```

In [0]:
resp = w.genie.list_spaces()
for space in resp.spaces:
    if space.title.startswith("Velocity"):
        print(space.space_id, space.title)

01f0fe3570de1a69a78b1b9b71d392ff Velocity Motors - Unified Analytics
01f0fe356cbe15a7a4259f6822d1ebe8 Velocity Motors - Operations & Inventory
01f0fe356c771e348b333b15f82c2e15 Velocity Motors - Customer Intelligence
01f0fe356c511a278a49fdc1e83797e3 Velocity Motors - Sales Analytics


In [0]:
# Domain-specific Genie Space IDs (deployed by 00c_setup_genie)
# Replace these with your actual space IDs after running setup
DOMAIN_SPACES = [
    "01f0fe356cbe15a7a4259f6822d1ebe8",  # Velocity Motors - Sales Analytics
    "01f0fe356c771e348b333b15f82c2e15",  # Velocity Motors - Customer Intelligence
    "01f0fe356c511a278a49fdc1e83797e3",  # Velocity Motors - Operations & Inventory
]

multi_orch = GenieOrchestrator(
    llm=llm,
    space_ids=DOMAIN_SPACES,
    client=w,
)

print(f"Multi-Genie orchestrator initialized with {len(multi_orch.space_ids)} spaces")

Multi-Genie orchestrator initialized with 3 spaces


### Router in Action

The router now reads metadata from **all 3 spaces** before deciding where to send each question. Watch the `Routed to space:` output — different questions should land on different spaces.

In [0]:
# Sales domain — should route to Sales Analytics space
multi_orch.execute("What are the top 5 selling vehicle models by total revenue?")

Routed to space: Velocity Motors - Sales Analytics (01f0fe356c511a278a49fdc1e83797e3)
Rewritten query: In the Velocity Motors data, identify the top 5 vehicle models by total revenue. Use the sales/orders data to compute revenue per vehicle model by summing order totals for completed orders, grouped by vehicle model, and sort descending to return the top 5. Include model, make, year, and total revenue. If total revenue requires joining vehicles, orders, and order_items (where item_type = 'Vehicle'), ensure calculation uses order_total or sum of item totals accordingly. Consider filtering for valid orders (status = 'Completed' or similar final state) and constrain to vehicles currently available for sale if relevant.

Question:
  What are the top 5 selling vehicle models by total revenue?

SQL:
  SELECT v.model, v.make, v.year, SUM(oi.total_price) AS total_revenue
FROM `sandbox`.`sales`.`vehicles` v
JOIN `sandbox`.`sales`.`orders` o ON o.vehicle_id = v.vehicle_id
JOIN `sandbox`.`sales`.

{'message': GenieMessage(id='01f0fe4876cb1f1c81b5f1a6446b07ac', space_id='01f0fe356c511a278a49fdc1e83797e3', conversation_id='01f0fe4876bf1c31b9102b85ff2ee2c4', content="In the Velocity Motors data, identify the top 5 vehicle models by total revenue. Use the sales/orders data to compute revenue per vehicle model by summing order totals for completed orders, grouped by vehicle model, and sort descending to return the top 5. Include model, make, year, and total revenue. If total revenue requires joining vehicles, orders, and order_items (where item_type = 'Vehicle'), ensure calculation uses order_total or sum of item totals accordingly. Consider filtering for valid orders (status = 'Completed' or similar final state) and constrain to vehicles currently available for sale if relevant.", message_id='01f0fe4876cb1f1c81b5f1a6446b07ac', attachments=[GenieAttachment(attachment_id='01f0fe48793e1e7c99e8daf1a4ec2d50', query=GenieQueryAttachment(description='You want to see the top 5 vehicle model

Trace(trace_id=tr-c5eb3e312f036c8ebdcd37023ff545c5)

In [0]:
# CRM domain — should route to Customer Intelligence space
multi_orch.execute("Which customer segments generate the most service revenue?")

Routed to space: Velocity Motors - Customer Intelligence (01f0fe356c771e348b333b15f82c2e15)
Rewritten query: From the Velocity Motors data, identify which customer segments generate the most service revenue. Please return: segment_id, segment_name, total_service_revenue (sum of service_orders.total_cost where service_orders joined to customers via customers.customer_id -> customers.segment_id), total_services_count, average_service_value. Filter to completed services only, and consider the service_type breakdown if helpful. Include timeframe parameter (e.g., last 12 months) optional. Ensure joins: sandbox.crm.customers (customer_id, segment_id) -> sandbox.crm.customer_segments (segment_id, segment_name) and sandbox.operations.service_orders (service_order_id, customer_id, total_cost, service_date, status). Group by segment_id, segment_name. Order by total_service_revenue desc, limit 10.

Question:
  Which customer segments generate the most service revenue?

Answer:
  It appears you ar

{'message': GenieMessage(id='01f0fe487fb6139da09b21f5e24828a9', space_id='01f0fe356c771e348b333b15f82c2e15', conversation_id='01f0fe487fa71b06be1b97b47dca44e7', content='From the Velocity Motors data, identify which customer segments generate the most service revenue. Please return: segment_id, segment_name, total_service_revenue (sum of service_orders.total_cost where service_orders joined to customers via customers.customer_id -> customers.segment_id), total_services_count, average_service_value. Filter to completed services only, and consider the service_type breakdown if helpful. Include timeframe parameter (e.g., last 12 months) optional. Ensure joins: sandbox.crm.customers (customer_id, segment_id) -> sandbox.crm.customer_segments (segment_id, segment_name) and sandbox.operations.service_orders (service_order_id, customer_id, total_cost, service_date, status). Group by segment_id, segment_name. Order by total_service_revenue desc, limit 10.', message_id='01f0fe487fb6139da09b21f5e

Trace(trace_id=tr-7d9b137c2b58db02a792ba67495281af)

In [0]:
# Operations domain — should route to Operations & Inventory space
multi_orch.execute("Which parts are below their reorder point?")

Routed to space: Velocity Motors - Operations & Inventory (01f0fe356cbe15a7a4259f6822d1ebe8)
Rewritten query: Identify all parts where the current quantity_on_hand is below the reorder_point. Return part_id, part_number, part_name, category, quantity_on_hand, reorder_point, and status for those parts. If possible, include the warehouse_id and last_restocked_date to provide context on restocking urgency.

Question:
  Which parts are below their reorder point?

SQL:
  SELECT part_id, part_number, part_name, category, quantity_on_hand, reorder_point, status, warehouse_id, last_restocked_date
FROM `sandbox`.`operations`.`parts_inventory`
WHERE quantity_on_hand IS NOT NULL AND reorder_point IS NOT NULL AND quantity_on_hand < reorder_point
ORDER BY quantity_on_hand ASC

Answer:
  There are **292 parts** where the current quantity on hand is below the reorder point, indicating a need for restocking. Notable data points include:
- **Front Bumper** (BOD-5333C): 1 on hand, reorder point 3, statu

{'message': GenieMessage(id='01f0fe4885c61d4383fb86fc17940779', space_id='01f0fe356cbe15a7a4259f6822d1ebe8', conversation_id='01f0fe4885bc11c7b3b4c3466ef060cd', content='Identify all parts where the current quantity_on_hand is below the reorder_point. Return part_id, part_number, part_name, category, quantity_on_hand, reorder_point, and status for those parts. If possible, include the warehouse_id and last_restocked_date to provide context on restocking urgency.', message_id='01f0fe4885c61d4383fb86fc17940779', attachments=[GenieAttachment(attachment_id='01f0fe48883e143bba03995c1bf26aba', query=GenieQueryAttachment(description='You want to see all parts where the current stock quantity is below the reorder threshold, including details like part ID, number, name, category, current quantity, reorder point, status, warehouse location, and last restock date to understand restocking urgency.', id=None, last_updated_timestamp=None, parameters=[], query='SELECT part_id, part_number, part_name,

Trace(trace_id=tr-29ef2b1b1d1fb6f393e4294d73bcebd7)

### Cross-Domain Questions

The real power of multi-space routing: ask an ambiguous question that could touch multiple domains. The router picks the best space based on table metadata.

In [0]:
# Cross-domain — router must decide: is this CRM, Sales, or Operations?
multi_orch.execute("What is the customer lifetime value including service history?")

Routed to space: Velocity Motors - Customer Intelligence (01f0fe356c771e348b333b15f82c2e15)
Rewritten query: Compute the Customer Lifetime Value (LTV) per customer, including service history. Provide: (1) total lifetime_value from sandbox.crm.customers for each customer_id, (2) total revenue generated from all service orders in sandbox.sales.orders and line items in sandbox.sales.order_items linked to those customers (including total order value and discounts), (3) a breakdown of service history by service_type from sandbox.operations.service_orders (counts, total_cost, and average customer rating per service_type), and (4) a joint summary by customer showing: customer_id, customer_name, lifetime_value, total_service_revenue, number_of_services, average_service_value, and latest_service_date. Filter by customers with at least one completed service and/or lifetime_value > 0, and include a time horizon of the last 24 months for service orders. Ensure joins use: customers.customer_id -> s

{'message': GenieMessage(id='01f0fe4890711d058fa2cb93911a6b6d', space_id='01f0fe356c771e348b333b15f82c2e15', conversation_id='01f0fe48906a142bbe8bd79b36426aff', content='Compute the Customer Lifetime Value (LTV) per customer, including service history. Provide: (1) total lifetime_value from sandbox.crm.customers for each customer_id, (2) total revenue generated from all service orders in sandbox.sales.orders and line items in sandbox.sales.order_items linked to those customers (including total order value and discounts), (3) a breakdown of service history by service_type from sandbox.operations.service_orders (counts, total_cost, and average customer rating per service_type), and (4) a joint summary by customer showing: customer_id, customer_name, lifetime_value, total_service_revenue, number_of_services, average_service_value, and latest_service_date. Filter by customers with at least one completed service and/or lifetime_value > 0, and include a time horizon of the last 24 months for

Trace(trace_id=tr-8177a6e1ee693be948d1000c56f8f1e5)

## Cross-Domain Synthesis

The router picks **one** space per question. But some questions span multiple domains:

> _"Compare revenue by region with customer satisfaction ratings and parts inventory levels"_

No single Genie Space has all the data. The solution: **decompose → fan-out → synthesize**.

```
Complex Question
       |
       v
   Decomposer (LLM)
       |
       +---> Sub-query 1 → Sales Genie      → revenue by region
       +---> Sub-query 2 → CRM Genie        → satisfaction ratings
       +---> Sub-query 3 → Operations Genie  → inventory levels
       |
       v
   Synthesizer (LLM)
       |
       v
   Unified Answer
```

In [0]:
class DecomposerResponse(BaseModel):
    """Structured response from the query decomposer LLM."""
    sub_queries: list[RouterResponse] = Field(description="List of focused sub-queries, each targeting exactly one Genie Space")

def ask_decomposer(
    llm: ChatDatabricks,
    question: str,
    genie_spaces: list[str],
    client: WorkspaceClient = None,
) -> list[dict[str, str]]:
    """Break a complex question into sub-queries, each targeted at a specific Genie Space.

    Returns a list of {"query": "...", "space_id": "..."} dicts.
    """
    prompt = f"""
You are a query decomposer for Databricks Genie AI. You are given a complex question
that may require data from multiple Genie Spaces (each space covers a different data domain).

Your job is to break the question into focused sub-queries, each targeting exactly one Genie Space.
Each sub-query should be self-contained and answerable by a single space.

Rules:
- Each sub-query must target exactly one space_id from the list below
- Keep sub-queries simple and specific — Genie works best with focused questions
- Use 1-4 sub-queries (only as many as needed)
- If the question only needs one space, return a single-element list

Question: {question}
"""
    prompt += "\n\nGenie Spaces Available and their metadata:\n"
    for space_id in genie_spaces:
        space_metadata = GenieUtils.get_space_metadata(space_id, client=client, enrich_columns=True)
        prompt += f"Space ID: {space_id}\nMetadata: {space_metadata}\n\n"

    # Use structured output to guarantee valid JSON
    structured_llm = llm.with_structured_output(DecomposerResponse)
    result = structured_llm.invoke(prompt)

    return [sq.model_dump() for sq in result.sub_queries]


In [0]:
def execute_multi(self, user_question: str, output_format: str = "full"):
    """Decompose a complex question across multiple spaces and synthesize results.

    1. Decomposer LLM breaks the question into targeted sub-queries
    2. Each sub-query runs against its assigned Genie Space
    3. Synthesizer LLM combines all results into a unified answer

    Args:
        output_format: "full", "text", "llm", or "raw" (same as execute)
    """
    # Step 1: Decompose
    print(f"Decomposing: {user_question}\n")
    sub_queries = ask_decomposer(
        llm=self.llm,
        question=user_question,
        genie_spaces=self.space_ids,
        client=self.client,
    )

    print(f"Plan: {len(sub_queries)} sub-queries")
    for i, sq in enumerate(sub_queries, 1):
        title = self.space_titles.get(sq["space_id"], sq["space_id"])
        print(f"  {i}. [{title}] {sq['query']}")
    print()

    # Step 2: Fan-out — execute each sub-query
    results = []
    for i, sq in enumerate(sub_queries, 1):
        title = self.space_titles.get(sq["space_id"], sq["space_id"])
        print(f"Running sub-query {i}/{len(sub_queries)}: {title}...")

        try:
            response = GenieUtils.ask_genie(
                query=sq["query"],
                space_id=sq["space_id"],
                client=self.client,
            )
            results.append(
                {
                    "space_id": sq["space_id"],
                    "space_title": title,
                    "query": sq["query"],
                    "sql": response.get("sql"),
                    "text": response.get("text"),
                    "success": True,
                }
            )
            print("  Done.\n")
        except Exception as e:
            results.append(
                {
                    "space_id": sq["space_id"],
                    "space_title": title,
                    "query": sq["query"],
                    "sql": None,
                    "text": None,
                    "success": False,
                    "error": str(e),
                }
            )
            print(f"  Failed: {e}\n")

    # Step 3: Synthesize
    synthesis_context = "\n\n".join(
        f"--- {r['space_title']} ---\nQuery: {r['query']}\nAnswer: {r.get('text', 'No answer')}" for r in results
    )

    synthesis_prompt = f"""You are a data analyst synthesizer. You were given a complex question that was
broken into sub-queries across different data domains. Below are the results from each domain.

Original question: {user_question}

Sub-query results:
{synthesis_context}

Produce a unified, well-structured answer that combines insights from all domains.
Be specific with numbers and highlight cross-domain patterns."""

    synthesis = self.llm.invoke(synthesis_prompt)

    if output_format == "full":
        print(f"Question:\n  {user_question}\n")
        for r in results:
            print(f"[{r['space_title']}]")
            if r.get("sql"):
                print(f"  SQL: {r['sql']}")
            if r.get("text"):
                print(f"  Result: {r['text']}")
            print()
        print(f"Synthesized Answer:\n  {synthesis.content}")
    elif output_format == "text":
        print(synthesis.content)
    elif output_format == "llm":
        display(Markdown(synthesis.content))
    # "raw" — no printing

    return {
        "question": user_question,
        "sub_queries": sub_queries,
        "results": results,
        "synthesis": synthesis.content,
    }


# Patch onto GenieOrchestrator
GenieOrchestrator.execute_multi = execute_multi

### Try It

These questions intentionally span multiple domains — the decomposer should create sub-queries for 2–3 different spaces.

In [0]:
# Spans Sales + CRM + Operations
multi_orch.execute_multi(
    "Compare revenue by region with customer satisfaction ratings and parts inventory levels",
    output_format="llm",
)

Decomposing: Compare revenue by region with customer satisfaction ratings and parts inventory levels

Plan: 3 sub-queries
  1. [Velocity Motors - Sales Analytics] From Velocity Motors - Sales Analytics, compute total revenue by region
  2. [Velocity Motors - Operations & Inventory] From Velocity Motors - Operations & Inventory, obtain average customer satisfaction ratings for completed services by region
  3. [Velocity Motors - Operations & Inventory] From Velocity Motors - Operations & Inventory, fetch current parts inventory levels by region or warehouse location

Running sub-query 1/3: Velocity Motors - Sales Analytics...
  Done.

Running sub-query 2/3: Velocity Motors - Operations & Inventory...
  Done.

Running sub-query 3/3: Velocity Motors - Operations & Inventory...
  Done.



Here is a unified view that combines revenue by region, customer satisfaction, and parts inventory levels for Velocity Motors. I’ve highlighted cross-domain patterns and paired the exact figures where relevant.

1) Revenue by region (Velocity Motors – Sales Analytics)
- West region: $995,166,736 (highest)
- Northeast region: $849,298,338
- Pacific Northwest: $390,965,087 (lowest)
- Pattern: Revenue varies widely by region, with the West leading by a substantial margin over the others (roughly $146M more than Northeast, and about $604M more than Pacific Northwest).

2) Customer satisfaction ratings (completed services, Velocity Motors – Operations & Inventory)
- Rhode Island: 4.31 (259 ratings)
- Iowa: 4.31 (238 ratings)
- Florida: 4.30 (214 ratings)
- Idaho: 4.30 (241 ratings)
- Hawaii: 4.29 (229 ratings)
- Pattern: Overall high satisfaction across multiple regions, with Rhode Island and Iowa tied for the top average (4.31). The ratings are near the upper end of typical service scales, suggesting strong customer experience for completed services in these regions.

3) Current parts inventory levels by warehouse/location (Velocity Motors – Operations & Inventory)
- Adamsland Parts Depot (VA): 21,439 parts (highest)
- South Susan Parts Depot (NC): 21,044 parts
- Juarezfort Parts Depot (NY): 20,032 parts
- Greenbury Regional Warehouse (TX): 15,828 parts (lowest)
- Range: 15,828 to 21,439 parts across listed locations
- Pattern: Inventory levels are relatively high at multiple depots with a spread of about 5,611 parts between the highest and lowest locations. No explicit regional ties to revenue are given for these inventory counts, but the largest stock is at Adamsland (VA) and South Susan (NC), which could align with higher service volumes in those corridors if demand signals concur.

Cross-domain patterns and insights
- Revenue dominance by West region (nearly $995M) contrasts with the regional satisfaction data provided, which lists top satisfaction in Rhode Island, Iowa, Florida, Idaho, and Hawaii. This suggests that high revenue regions (West, Northeast) do not necessarily coincide with the top satisfaction points in the given sample. Specifically:
  - West region revenue is the highest, but the satisfaction data points do not explicitly chart by region on the West or Northeast. If satisfaction data were extended by region, we could assess whether high revenue regions also maintain solid satisfaction or if there are gaps to address.
- Inventory levels show substantial stock at several East Coast and adjacent locations (VA, NC, NY), with a noticeably lower stock in TX. If the West/Northeast regions align with these locations, there may be a need to map warehouse regions to sales regions to evaluate whether stock levels support revenue performance and delivery times in high-revenue areas.
- The top satisfaction scores (Rhode Island and Iowa) are on the lower end of the inventory list’s geographic spread (Rhode Island is not among the highest inventory locations listed here, and Iowa is not explicitly listed as a warehouse; the data may reflect broader operations beyond the top three inventory points). This could indicate that high satisfaction is achieved despite variable inventory levels, or it could reflect effective service fulfillment from nearby inventory sources.

Actionable synthesis
- To better align inventory with revenue and satisfaction:
  - Map each region’s revenue to the nearest inventory locations and assess stock-to-demand ratios. For example, if West region corresponds to locations not listed among the top inventory centers, consider whether replenishment is adequate to support high-volume sales.
  - Compare parts availability against service completion satisfaction in overlapping regions (e.g., if Rhode Island and Iowa have high satisfaction but are not identified as major revenue regions, confirm whether inventory constraints exist there or if service teams operate efficiently with current stock).
- Data gaps to consider collecting next:
  - Region-to-warehouse mapping to link inventory levels to regional demand and service ratings.
  - Region-level satisfaction broken out by each region’s sales performance, including West, Northeast, Pacific Northwest, etc., to directly compare with revenue figures.
  - Inventory turnover or days-of-supply by warehouse to understand whether high-stock locations (VA, NC, NY) translate into faster fulfillment in high-revenue regions.

Key numbers recap
- West revenue: $995,166,736
- Northeast revenue: $849,298,338
- Pacific Northwest revenue: $390,965,087
- Top satisfaction (region data points): Rhode Island 4.31, Iowa 4.31
- Other satisfaction points: Florida 4.30, Idaho 4.30, Hawaii 4.29
- Inventory highlights: Adamsland VA 21,439; South Susan NC 21,044; Juarezfort NY 20,032; Greenbury TX 15,828

If you’d like, I can create a data fusion plan or draft a set of queries to align regions, inventory, and satisfaction (e.g., regional mapping table, inventory-to-region linkage, and a computed score combining revenue, satisfaction, and stock availability).

{'question': 'Compare revenue by region with customer satisfaction ratings and parts inventory levels',
 'sub_queries': [{'query': 'From Velocity Motors - Sales Analytics, compute total revenue by region',
   'space_id': '01f0fe356c511a278a49fdc1e83797e3'},
  {'query': 'From Velocity Motors - Operations & Inventory, obtain average customer satisfaction ratings for completed services by region',
   'space_id': '01f0fe356cbe15a7a4259f6822d1ebe8'},
  {'query': 'From Velocity Motors - Operations & Inventory, fetch current parts inventory levels by region or warehouse location',
   'space_id': '01f0fe356cbe15a7a4259f6822d1ebe8'}],
 'results': [{'space_id': '01f0fe356c511a278a49fdc1e83797e3',
   'space_title': 'Velocity Motors - Sales Analytics',
   'query': 'From Velocity Motors - Sales Analytics, compute total revenue by region',
   'sql': "SELECT\n  s.region,\n  SUM(oi.total_price) AS revenue\nFROM `sandbox`.`sales`.`orders` o\nJOIN `sandbox`.`sales`.`salespersons` s ON o.salesperson_id =

[Trace(trace_id=tr-aa3c61a38dbceee118af4a4ec362d8d1), Trace(trace_id=tr-8f86949afdcff3b1d9c14464e38d3c56)]

In [0]:
# Spans Sales + Operations
multi_orch.execute_multi(
    "Which vehicle models have the highest service costs relative to their sale price?",
    output_format="llm",
)

Decomposing: Which vehicle models have the highest service costs relative to their sale price?

Plan: 1 sub-queries
  1. [Velocity Motors - Operations & Inventory] Within Velocity Motors - Operations & Inventory, identify vehicle models along with their sale price and service cost so we can compare service cost relative to sale price.

Running sub-query 1/1: Velocity Motors - Operations & Inventory...
  Done.



Below is a unified synthesis across domains, focusing on which vehicle models incur the highest service costs relative to their sale prices. I combine the provided Velocity Motors data with cross-domain patterns and highlight concrete numbers and takeaways.

1) Key finding: service cost relative to sale price can vary widely within the same model line
- From Velocity Motors - Operations & Inventory, several entries for the 3 Series show a wide spread in service costs at different sale prices:
  - Sale price $20,956 → average service cost $58.04
  - Sale price $21,003 → average service cost $456.43
  - Sale price $22,388 → average service cost $708.44
  - Sale price $22,971 → average service cost $93.73
  - Sale price $23,605 → average service cost $136.54
- This indicates that even within the same model family (3 Series), the service cost does not scale predictably with sale price. Some higher-priced examples have modest service costs, while others (even at similar price points) show very high service costs (e.g., $708.44 at $22,388).

2) Cross-domain patterns to monitor when ranking by “highest service cost relative to sale price”
- Absolute service cost vs. sale price:
  - Absolute service costs range widely (e.g., $58.04 to $708.44 in the provided 3 Series samples).
  - The highest absolute service cost in the sample is $708.44 at $22,388 sale price.
  - However, the highest relative cost (service cost as a percentage of sale price) could differ. For example:
    - $58.04 on $20,956 sale price ≈ 0.277%
    - $456.43 on $21,003 ≈ 2.18%
    - $708.44 on $22,388 ≈ 3.16%
    - $93.73 on $22,971 ≈ 0.41%
    - $136.54 on $23,605 ≈ 0.58%
  - Relative costs show that the $22,388 sale price with $708.44 service cost is the highest relative burden among the examples (≈3.16%).

3) Implications for “highest service costs relative to sale price” (ranked insights)
- The 3 Series at sale price $22,388 with average service cost $708.44 delivers the strongest signal of high relative service cost (≈3.16% of sale price).
- The next strong relative case is $21,003 sale price with $456.43 service cost (≈2.18%).
- The other 3 Series observations show much lower relative costs (sub-1% ranges).

4) Recommendations for a complete, robust answer (beyond the provided 5 data points)
- Expand the dataset to compute exact service-cost-to-sale-price ratio for every model-price pair, then rank by the ratio.
- Compute two metrics per model:
  - Absolute service cost (to identify highest dollar amounts)
  - Relative service cost (service cost / sale price) to identify the highest burden relative to price
- Ensure deduplication or proper aggregation:
  - If multiple entries exist for the same model at the same sale price, decide whether to average the service costs or take the maximum.
  - If multiple sale prices exist for the same model (e.g., different trims or configurations), compute ratios per price point and then compute an overall maximum ratio per model or a trimmed mean.
- Cross-domain comparison:
  - If other domains (e.g., maintenance history, region, vehicle age, total ownership cost) are available, incorporate them to see if high relative service cost correlates with:
    - Higher-mileage fleets
    - Certain regions with higher service pricing
    - Specific trims or configurations within the same model family

5) Preliminary top candidates based on provided data
- Based solely on the Velocity Motors snippet:
  - 3 Series at sale price $22,388 with service cost $708.44 (relative cost ≈ 3.16%) appears to be the top relative-cost example.
  - 3 Series at sale price $21,003 with service cost $456.43 (≈ 2.18%) is the next highest relative-cost case among the listed samples.
  - The $20,956 price point with $58.04 is the lowest absolute service cost and also a very small relative cost (≈0.28%).
- Note: This ranking is constrained to the five listed data points; a full ranking requires the complete dataset.

6) Actionable next steps
- Provide the full dataset of Velocity Motors - Operations & Inventory (all model-price pairs with their average service costs) to compute an authoritative ranking.
- If available, share cross-domain data (e.g., average maintenance hours, parts costs, warranty coverage, model year, mileage) to contextualize service costs.
- Deliver a final report that includes:
  - A ranked list of models by highest service-cost-to-sale-price ratio
  - Both absolute and relative cost views
  - Clear notes on data limitations (e.g., sample size, potential duplicates)

In summary, from the supplied data, the standout case for “highest service costs relative to sale price” is the 3 Series at a $22,388 sale price with $708.44 in average service cost (relative approx. 3.16%). However, a complete answer requires computing the ratios across the full dataset and across all models, then aggregating in a consistent way. If you can share the full dataset or confirm the preferred aggregation method (average vs. max per price point), I can produce a definitive ranked list with exact percentages and a concise executive summary.

{'question': 'Which vehicle models have the highest service costs relative to their sale price?',
 'sub_queries': [{'query': 'Within Velocity Motors - Operations & Inventory, identify vehicle models along with their sale price and service cost so we can compare service cost relative to sale price.',
   'space_id': '01f0fe356cbe15a7a4259f6822d1ebe8'}],
 'results': [{'space_id': '01f0fe356cbe15a7a4259f6822d1ebe8',
   'space_title': 'Velocity Motors - Operations & Inventory',
   'query': 'Within Velocity Motors - Operations & Inventory, identify vehicle models along with their sale price and service cost so we can compare service cost relative to sale price.',
   'sql': 'SELECT v.model, v.msrp, ROUND(AVG(s.total_cost), 2) AS avg_service_cost\nFROM `sandbox`.`sales`.`vehicles` v\nJOIN `sandbox`.`operations`.`service_orders` s ON v.vehicle_id = s.vehicle_id\nWHERE v.model IS NOT NULL AND v.msrp IS NOT NULL AND s.total_cost IS NOT NULL\nGROUP BY v.model, v.msrp\nORDER BY v.model, v.msrp',
  

[Trace(trace_id=tr-f3530a78a5980b02c27fd0c8506e3f43), Trace(trace_id=tr-c72b6c140481dc10b359ecd1987ff555)]

---
## Summary

| What you did | Code |
|---|---|
| Connect to Genie | `WorkspaceClient()` + `genie.start_conversation_and_wait()` |
| Parse response | Loop over `message.attachments` |
| Get data rows | `GenieUtils.to_dataframe(result, client)` → `pandas.DataFrame` |
| Follow-up | `genie.create_message_and_wait()` with `conversation_id` |
| Verbose polling | `GenieUtils.ask_genie_verbose()` — watch status transitions |
| Space metadata | `GenieUtils.get_space_metadata()` — table schemas + sample questions |
| LLM routing | `ask_router()` — rewrite question + pick target space |
| Single-space orchestration | `GenieOrchestrator.execute()` — router → Genie → format |
| Multi-space routing | Pass multiple space IDs → router picks the best one |
| Cross-domain synthesis | `GenieOrchestrator.execute_multi()` — decompose → fan-out → synthesize |

**The entire Genie API boils down to 3 SDK calls:**
1. `start_conversation_and_wait` — new question
2. `create_message_and_wait` — follow-up
3. `get_message_attachment_query_result` — get the data